# URL phishing classifier (GPT-2 small fine-tune)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import tiktoken
import torch
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = next(
    root
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "components").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "model"

SEED = 123
torch.manual_seed(SEED)

tokenizer = tiktoken.get_encoding("gpt2")
PAD_ID = 50256
MAX_LEN = 128  
LABEL_NAMES = {0: "Phishing", 1: "Legitimate"}

## Data

In [ ]:
data_file_path = DATA_DIR / "PhiUSIIL_Phishing_URL_Dataset.csv"
df = pd.read_csv(data_file_path)

data = df[["URL", "label"]].drop_duplicates(subset="URL")

print(data["label"].value_counts())
print(data["label"].unique())

In [ ]:
def balanced_subset(df, n_total, label_col="label", random_state=SEED):
    per_class = min(n_total // 2, df[label_col].value_counts().min())
    parts = [
        df[df[label_col] == label].sample(n=per_class, random_state=random_state)
        for label in (0, 1)
    ]
    return (
        pd.concat(parts)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


subset_df = balanced_subset(data, 20_000)
print(subset_df["label"].value_counts())  

# 70 / 10 / 20
train_df, rest_df = train_test_split(
    subset_df, train_size=0.7, stratify=subset_df["label"], random_state=SEED
)
validation_df, test_df = train_test_split(
    rest_df, train_size=1 / 3, stratify=rest_df["label"], random_state=SEED
)
print(len(train_df), len(validation_df), len(test_df))

In [ ]:
train_df.to_csv(DATA_DIR / "train.csv", index=False)
validation_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

In [ ]:
class PhishingDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=MAX_LEN):
        self.data = pd.read_csv(csv_file)
        self.max_length = max_length

        self.encoded_texts = [
            tokenizer.encode(url)[:max_length]
            for url in self.data["URL"]
        ]
        self.labels = self.data["label"].tolist()

    def __getitem__(self, idx):
        return self.encoded_texts[idx], self.labels[idx]

    def __len__(self):
        return len(self.labels)


def collate_pad(batch, pad_token_id=PAD_ID):
    encoded_texts, labels = zip(*batch)
    longest = max(len(e) for e in encoded_texts)
    input_ids = torch.full((len(encoded_texts), longest), pad_token_id, dtype=torch.long)
    for i, encoded in enumerate(encoded_texts):
        input_ids[i, :len(encoded)] = torch.tensor(encoded, dtype=torch.long)
    return input_ids, torch.tensor(labels, dtype=torch.long)

In [ ]:
train_dataset = PhishingDataset(
    csv_file=DATA_DIR / "train.csv",
    max_length=MAX_LEN,
    tokenizer=tokenizer,
)

val_dataset = PhishingDataset(
    csv_file=DATA_DIR / "validation.csv",
    max_length=MAX_LEN,
    tokenizer=tokenizer,
)

test_dataset = PhishingDataset(
    csv_file=DATA_DIR / "test.csv",
    max_length=MAX_LEN,
    tokenizer=tokenizer,
)

num_workers = 0
batch_size = 32
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
    collate_fn=collate_pad,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
    collate_fn=collate_pad,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
    collate_fn=collate_pad,
    pin_memory=pin_memory,
)

In [ ]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)  
print("Label batch dimensions", target_batch.shape)
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

## Model

In [ ]:
from components.gpt.load_gpt2 import download_and_load_gpt2, load_weights_into_gpt
from components.gpt.model import GPTModel
from components.config import MODEL, MODEL_CONFIG

model_size = MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="../model/gpt2")
model = GPTModel(MODEL_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
for p in model.parameters():
    p.requires_grad = False

num_classes = 2
model.out_head = torch.nn.Linear(
    in_features=MODEL_CONFIG["emb_dim"],
    out_features=num_classes
)

for p in model.trf_blocks[-1].parameters():
    p.requires_grad = True
for p in model.final_norm.parameters():
    p.requires_grad = True

print(model.out_head)

## Loss and accuracy helpers

In [ ]:
def last_real_token_logits(model, input_batch, pad_id=PAD_ID):
    with torch.autocast(
        device_type=input_batch.device.type,
        dtype=torch.bfloat16,
        enabled=input_batch.is_cuda,
    ):
        logits = model(input_batch)
    last_idx = (input_batch != pad_id).sum(dim=1).clamp(min=1) - 1
    return logits[torch.arange(logits.shape[0], device=logits.device), last_idx].float()


def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device, non_blocking=True)
            target_batch = target_batch.to(device, non_blocking=True)
            with torch.no_grad():
                logits = last_real_token_logits(model, input_batch)
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += ((predicted_labels == target_batch).sum().item())
        else:
            break
    return correct_predictions / num_examples


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)
    logits = last_real_token_logits(model, input_batch)
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

## Baseline

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_accuracy = calc_accuracy_loader(
    train_loader, model, device, num_batches=10
)
val_accuracy = calc_accuracy_loader(
    val_loader, model, device, num_batches=10
)
test_accuracy = calc_accuracy_loader(
    test_loader, model, device, num_batches=10
)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

In [ ]:
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

## Training

In [ ]:
@torch.inference_mode()
def evaluate_loader(data_loader, model, device, num_batches=None):
    model.eval()
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    total_loss = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    num_examples = 0

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        input_batch = input_batch.to(device, non_blocking=True)
        target_batch = target_batch.to(device, non_blocking=True)
        logits = last_real_token_logits(model, input_batch)
        total_loss += torch.nn.functional.cross_entropy(logits, target_batch, reduction="sum")
        correct += (logits.argmax(dim=-1) == target_batch).sum()
        num_examples += target_batch.shape[0]

    num_examples = max(num_examples, 1)
    return total_loss.item() / num_examples, correct.item() / num_examples


def train_classifier(model, train_loader, val_loader,
                            optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1
    best_val_accuracy, best_state = -1.0, None

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, _ = evaluate_loader(train_loader, model, device, eval_iter)
                val_loss, _ = evaluate_loader(val_loader, model, device, eval_iter)
                model.train()
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}")

        _, train_accuracy = evaluate_loader(train_loader, model, device, eval_iter)
        _, val_accuracy = evaluate_loader(val_loader, model, device) 
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {
                n: p.detach().cpu().clone()
                for n, p in model.named_parameters() if p.requires_grad
            }

    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        print(f"Restored best checkpoint (val accuracy {best_val_accuracy*100:.2f}%)")

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
import time

use_fused = device.type == "cuda"

trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
head_params = [p for n, p in trainable if n.startswith("out_head")]
body_params = [p for n, p in trainable if not n.startswith("out_head")]

optimizer = torch.optim.AdamW(
    [
        {"params": head_params, "lr": 1e-3},   
        {"params": body_params, "lr": 5e-5},  
    ],
    weight_decay=0.1,
    fused=use_fused,
)

num_epochs = 3
start_time = time.perf_counter()
train_losses, val_losses, train_accs, val_accs, examples_seen = \
    train_classifier(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=num_epochs, eval_freq=100,
        eval_iter=10
    )
execution_time_minutes = (time.perf_counter() - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

## Evaluation

In [ ]:
@torch.no_grad()
def predict_loader(data_loader, model, device):
    model.eval()
    preds, labels = [], []
    for input_batch, target_batch in data_loader:
        logits = last_real_token_logits(model, input_batch.to(device))
        preds.append(logits.argmax(dim=-1).cpu())
        labels.append(target_batch)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


preds, labels = predict_loader(test_loader, model, device)
print(f"Test accuracy: {(preds == labels).mean()*100:.2f}% ({len(labels)} URLs)")
print(classification_report(
    labels, preds,
    target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
    digits=3,
))
print(confusion_matrix(labels, preds))  

In [ ]:
@torch.no_grad()
def classify_url(url, model, tokenizer, device, max_length=MAX_LEN):
    model.eval()
    ids = torch.tensor(tokenizer.encode(url)[:max_length], device=device).unsqueeze(0)
    probs = torch.softmax(last_real_token_logits(model, ids), dim=-1)[0]
    label = int(probs.argmax())
    return label, probs[label].item()


model.eval()
test_examples = pd.read_csv(DATA_DIR / "test.csv")

correct = 0
length_tested = 0

for true_label, url in zip(test_examples["label"][:10], test_examples["URL"][:10]):
    true_label = int(true_label)
    pred_label, confidence = classify_url(url, model, tokenizer, device)

    pred = LABEL_NAMES[pred_label]
    true = LABEL_NAMES[true_label]

    ok = pred_label == true_label
    correct += ok

    print(
        f"{'Correct:' if ok else 'Wrong:'} "
        f"true={true:<10} pred={pred:<10} ({confidence:.2f}) {url}"
    )
    length_tested += 1

print(
    f"\nSpot-check accuracy: {correct}/{length_tested} "
    f"= {correct / length_tested:.0%} (illustration only, see full test set above)"
)

In [ ]:
from components.inference import save_classifier
save_classifier(model)